# 🏦 Retail Lending Origination & Risk-Based Pricing Engine
## 75,000 Applicants | Dual Scoring | CBK KESONIA Pricing | Product Mix Optimization

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')
KESONIA_BASELINE = 0.0875
OP_COST_FACTOR = 0.025
FUNDS_COST = 0.010
PORTFOLIO_PD_CAP = 0.05
MONTHLY_CAPITAL = 100_000_000
print('✓ Engine initialized')


✓ Engine initialized


In [2]:
np.random.seed(42)
applicant_data = []
for i in range(75_000):
    product = np.random.choice(['Mobile_Micro', 'Micro_SME', 'Diaspora_Remittance'], p=[0.60, 0.25, 0.15])
    monthly_income = np.random.lognormal([10.5, 11.5, 11.2][{'Mobile_Micro': 0, 'Micro_SME': 1, 'Diaspora_Remittance': 2}[product]], 0.8)
    mpesa_inflow = monthly_income * np.random.uniform(0.5, 2.5)
    mpesa_outflow = monthly_income * np.random.uniform(0.6, 1.8)
    till_turnover = monthly_income * (np.random.uniform(1.0, 3.0) if product == 'Micro_SME' else np.random.uniform(0.1, 0.8))
    diaspora_remittance = monthly_income * (np.random.uniform(0.7, 1.5) if product == 'Diaspora_Remittance' else np.random.uniform(0.0, 0.3))
    crb_score = np.clip(np.random.normal([550, 600, 650][{'Mobile_Micro': 0, 'Micro_SME': 1, 'Diaspora_Remittance': 2}[product]], 100), 300, 850)
    dti_ratio = np.random.beta(2, 5)
    requested_amount = [np.random.uniform(5000, 100000), np.random.uniform(50000, 1000000), np.random.uniform(100000, 500000)][{'Mobile_Micro': 0, 'Micro_SME': 1, 'Diaspora_Remittance': 2}[product]]
    requested_term = np.random.choice([3, 6, 12, 24], p=[0.25, 0.35, 0.25, 0.15])
    default_prob = np.clip(0.08 + (700 - crb_score) / 1000 + dti_ratio * 0.05 + {'Mobile_Micro': 0.02, 'Micro_SME': -0.01, 'Diaspora_Remittance': -0.015}[product], 0.01, 0.25)
    applicant_data.append({'Product': product, 'Monthly_Income': monthly_income, 'MPesa_Inflow': mpesa_inflow, 'MPesa_Outflow': mpesa_outflow, 'Till_Turnover': till_turnover, 'Diaspora_Remittance': diaspora_remittance, 'CRB_Score': crb_score, 'DTI': dti_ratio, 'Loan_Amount': requested_amount, 'Loan_Term': requested_term, 'Default': 1 if np.random.random() < default_prob else 0})

df = pd.DataFrame(applicant_data)
print(f'✓ Data: {len(df):,} applicants | Default rate: {df["Default"].mean()*100:.2f}%')

✓ Data: 75,000 applicants | Default rate: 19.02%
